# Homework 2 — Part 1: Data Cleaning & Validation

This notebook loads the intentionally dirty `ohlc_hw` table, inspects it,
and detects data-quality issues.

Run it with this folder (`Homeworks/HW2/Philip_Szkudlarski/`) as the working
directory. The database is a local copy at `data/data.db` and is not committed.

## Setup
Imports, plus `clean_ticker`, which is used in both 1.1 and 1.2.
Some ticker labels arrive wrapped in HTML, e.g.
`<a ...><span style="color: red">AAPL</span></a>`; this helper strips that.

In [1]:
import numpy as np
import sqlite3
import pandas as pd
from IPython.display import display
import re
import html


def clean_ticker(value):
    """Clean a ticker symbol and remove embedded HTML."""
    if pd.isna(value):
        return pd.NA

    value = html.unescape(str(value))
    value = re.sub(r"<[^>]*>", "", value)
    value = value.strip().upper()

    return value if value else pd.NA

## 1.1 Load and Inspect the Data

Load `ohlc_hw`, plus the `ohlc` table's ticker/date pairs, which serve as the
reference trading calendar in 1.2.

The SQL `ORDER BY` sorts on the raw (HTML-wrapped) labels, so after cleaning
the tickers we sort again by ticker, then date (oldest to newest).

In [2]:
conn = sqlite3.connect("data/data.db")

df_reference = pd.read_sql_query("SELECT ts, ticker FROM ohlc", conn)


query = """
SELECT *
FROM ohlc_hw
ORDER BY ticker, ts;
"""

# parse_dates converts ts to datetime
df = pd.read_sql_query(query, conn, parse_dates=["ts"])

# Strip HTML from ticker labels, then re-sort: the SQL ORDER BY sorted the raw
# HTML-wrapped labels, so each ticker's rows were split across several groups
df["ticker"] = df["ticker"].map(clean_ticker)
df = df.sort_values(["ticker", "ts"]).reset_index(drop=True)
df["ticker"].unique()

array(['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX',
       'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM',
       'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ',
       'WMT'], dtype=object)

### Basic statistics

In [3]:
print("=" * 60)
print("DATAFRAME INFORMATION")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)
print(df.describe().to_string())

DATAFRAME INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   ts         397711 non-null  datetime64[ns]
 1   open       397711 non-null  float64       
 2   high       397711 non-null  float64       
 3   low        397711 non-null  float64       
 4   close      397711 non-null  float64       
 5   adj_close  377856 non-null  float64       
 6   volume     397711 non-null  int64         
 7   ticker     397711 non-null  object        
dtypes: datetime64[ns](1), float64(5), int64(1), object(1)
memory usage: 24.3+ MB

SUMMARY STATISTICS
                                  ts           open           high            low          close      adj_close        volume
count                         397711  397711.000000  397711.000000  397711.000000  397711.000000  377856.000000  3.977110e+05
mean   1998-04-17 21:18:45.2924861

### Columns and their meanings

In [4]:
print("\n" + "=" * 60)
print("COLUMN NAMES AND MEANINGS")
print("=" * 60)

column_meanings = {
    "ts": "Date of the stock observation",
    "open": "Opening price for the trading day",
    "high": "Highest price during the trading day",
    "low": "Lowest price during the trading day",
    "close": "Closing price for the trading day",
    "adj_close": "Closing price adjusted for corporate actions",
    "volume": "Number of shares traded",
    "ticker": "Stock ticker symbol"
}

for column in df.columns:
    print(f"{column}: {column_meanings.get(column, 'Meaning not documented')}")


COLUMN NAMES AND MEANINGS
ts: Date of the stock observation
open: Opening price for the trading day
high: Highest price during the trading day
low: Lowest price during the trading day
close: Closing price for the trading day
adj_close: Closing price adjusted for corporate actions
volume: Number of shares traded
ticker: Stock ticker symbol


### Date range

In [5]:
print("\n" + "=" * 60)
print("DATE RANGE")
print("=" * 60)
print("Earliest date:", df["ts"].min())
print("Latest date:  ", df["ts"].max())


DATE RANGE
Earliest date: 1962-01-02 00:00:00
Latest date:   2024-12-06 00:00:00


### Tickers

In [6]:
print("\n" + "=" * 60)
print("TICKERS")
print("=" * 60)

tickers = sorted(df["ticker"].dropna().unique())
print("Number of unique ticker values:", len(tickers))
print(tickers)


TICKERS
Number of unique ticker values: 30
['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']


In [7]:
# Audit of ticker normalization: every raw label in ohlc_hw and what it became.
# Read straight from the database, because df["ticker"] was already cleaned above.
label_audit = pd.read_sql_query(
    "SELECT ticker AS raw_label, COUNT(*) AS n_rows FROM ohlc_hw GROUP BY ticker", conn)
label_audit["clean_label"] = label_audit["raw_label"].map(clean_ticker)
label_audit["changed"] = label_audit["raw_label"] != label_audit["clean_label"]
label_audit = label_audit.sort_values(["clean_label", "changed", "raw_label"]).reset_index(drop=True)

n_changed = label_audit.loc[label_audit["changed"], "n_rows"].sum()
print(f"{len(label_audit)} distinct raw labels -> {label_audit['clean_label'].nunique()} tickers")
print(f"Rows whose label was changed: {n_changed:,} of {label_audit['n_rows'].sum():,}")

# Example: all raw variants of one ticker
display(label_audit)

# Per ticker: how many raw variants, and how many rows had an HTML-wrapped label
per_ticker = label_audit.groupby("clean_label").apply(
    lambda g: pd.Series({
        "raw_variants": len(g),
        "rows_plain": g.loc[~g["changed"], "n_rows"].sum(),
        "rows_relabeled": g.loc[g["changed"], "n_rows"].sum(),
    }), include_groups=False)
display(per_ticker)


150 distinct raw labels -> 30 tickers
Rows whose label was changed: 39,936 of 397,711


,raw_label,n_rows,clean_label,changed
0,AAPL,10747,AAPL,False
1,"<a target=""_blank""><span style=""color: blue"">A...",307,AAPL,True
2,"<a target=""_blank""><span style=""color: red"">AA...",318,AAPL,True
3,"<a target=""_blank""><span style=""font-weight: b...",306,AAPL,True
4,"<a target=""_blank""><span style=""font-weight: b...",264,AAPL,True
...,...,...,...,...
145,WMT,12894,WMT,False
146,"<a target=""_blank""><span style=""color: blue"">W...",329,WMT,True
147,"<a target=""_blank""><span style=""color: red"">WM...",359,WMT,True
148,"<a target=""_blank""><span style=""font-weight: b...",359,WMT,True


,raw_variants,rows_plain,rows_relabeled
clean_label,,,
AAPL,5,10747,1195
AMGN,5,10154,1136
AMZN,5,6725,733
AXP,5,12910,1350
BA,5,15234,1746
CAT,5,15352,1692
CRM,5,4981,544
CSCO,5,8465,962
CVX,5,15349,1724


### Preview

In [8]:
print("\n" + "=" * 60)
print("FIRST 10 ROWS")
print("=" * 60)
print(df.head(10).to_string(index=False))


FIRST 10 ROWS
        ts    open    high     low   close  adj_close    volume ticker
1980-12-12 0.12835 0.12835 0.12835 0.12835   0.098941 469034068   AAPL
1980-12-15 0.12165 0.12165 0.12165 0.12165   0.093779 175884971   AAPL
1980-12-16 0.11272 0.11272 0.11272 0.11272   0.086896 105728103   AAPL
1980-12-17 0.11551 0.11551 0.11551 0.11551   0.089046  86441684   AAPL
1980-12-18 0.11886 0.11886 0.11886 0.11886   0.091627  73449671   AAPL
1980-12-19 0.12612 0.12612 0.12612 0.12612   0.097220  48630447   AAPL
1980-12-22 0.13225 0.13225 0.13225 0.13225   0.101950  37363236   AAPL
1980-12-23 0.13784 0.13784 0.13784 0.13784   0.106250  46950444   AAPL
1980-12-24 0.14509 0.14509 0.14509 0.14509   0.111850  48003248   AAPL
1980-12-26 0.15848 0.15848 0.15848 0.15848   0.122170  55574451   AAPL


**Observations from 1.1**
- 397,711 rows, but this includes duplicate ticker/date rows (see 1.2).
- `adj_close` is the only column with missing values.
- `open` has a minimum of 0, which is not a valid price.
- `adj_close` reaches 10,442.51 while `close` never exceeds 605.40, which suggests bad adjusted prices.
- The maximum `volume` (9.28 billion shares) is far above the rest of the column.
- Tickers arrive in 150 raw forms: each of the 30 tickers appears plain and
  wrapped in 4 HTML styles. 39,936 rows (~10%) carry an HTML label; most of
  these are the only copy of that day, so labels are normalized, not dropped.
  The full raw → clean mapping is kept in `label_audit`.


## 1.2 Detect Data Quality Issues

### Shared helpers
`prepare_data` returns a standardized copy (clean tickers, datetime dates,
numeric prices). `make_issue` builds one row of the Data Quality Report.

In [9]:
REPORT_COLUMNS = [
    "ticker", "issue_type", "issue_date", "description", "severity"
]


def prepare_data(df):
    """Return a standardized copy without modifying the input."""
    df = df.copy()

    # Clean ticker labels
    df["ticker"] = df["ticker"].map(clean_ticker)

    # Convert dates
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")

    # Convert numeric columns
    for col in ["open", "high", "low", "close", "adj_close", "volume"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def make_issue(ticker, issue_type, issue_date, description, severity):
    """Create one standardized data-quality issue."""
    return {
        "ticker": ticker,
        "issue_type": issue_type,
        "issue_date": issue_date,
        "description": description,
        "severity": severity,
    }

### Missing values by column

In [10]:
#check missing values
def check_missing_values(df):
    """Count missing values and percentages for every column."""
    counts = df.isna().sum()

    return pd.DataFrame({
        "missing_count": counts,
        "missing_percent": (
            counts / len(df) * 100 if len(df) else 0
        ).round(2),
    }).sort_values("missing_count", ascending=False)

### Missing trading days
Two complementary checks:
- `detect_missing_trading_days` uses the `ohlc` table as the reference calendar.
- `detect_date_gaps` needs no second table: it flags any jump of more than
  5 calendar days between consecutive dates within a ticker
  (weekends and holidays never exceed 4).

In [11]:
#check for missing trading days by ticker using ohlc as the reference calendar
def detect_missing_trading_days(df_hw, df_reference):
    """
    Return one record per ticker with missing reference dates.
    Only compare within each ticker's observed homework history.
    """
    hw = prepare_data(df_hw)
    ref = prepare_data(df_reference)

    hw["ts"] = hw["ts"].dt.normalize()
    ref["ts"] = ref["ts"].dt.normalize()

    results = []

    for ticker, group in hw.dropna(subset=["ticker", "ts"]).groupby("ticker"):
        start = group["ts"].min()
        end = group["ts"].max()

        reference_dates = set(ref.loc[
            (ref["ticker"] == ticker)
            & ref["ts"].between(start, end),
            "ts"
        ].dropna())

        observed_dates = set(group["ts"].dropna())
        missing_dates = sorted(reference_dates - observed_dates)

        results.append({
            "ticker": ticker,
            "start_date": start,
            "end_date": end,
            "missing_trading_days": len(missing_dates),
            "missing_dates": missing_dates,
            "summary": (
                f"Ticker {ticker} is missing {len(missing_dates)} "
                f"trading days between {start.date()} and {end.date()}."
            ),
        })

    return pd.DataFrame(results, columns=[
        "ticker", "start_date", "end_date",
        "missing_trading_days", "missing_dates", "summary"
    ]).sort_values("missing_trading_days", ascending=False)

In [12]:
#check for large gaps between consecutive dates within each ticker (no reference table needed)
def detect_date_gaps(df, max_gap_days=5):
    """
    Return one row per gap longer than max_gap_days calendar days.
    Weekends and holidays create gaps of up to 4 days, so 5 is a safe default.
    """
    df = df.dropna(subset=["ticker", "ts"]).drop_duplicates(["ticker", "ts"])
    df = df.sort_values(["ticker", "ts"])

    df["prev_ts"] = df.groupby("ticker")["ts"].shift(1)
    df["gap_days"] = (df["ts"] - df["prev_ts"]).dt.days

    gaps = df.loc[df["gap_days"] > max_gap_days, ["ticker", "prev_ts", "ts", "gap_days"]]
    return gaps.rename(columns={"prev_ts": "gap_start", "ts": "gap_end"}).reset_index(drop=True)

### Duplicate rows
- 29,460 duplicate ticker/date pairs. Every pair is an exact copy, including
  the raw label (26,509 plain/plain, 2,951 HTML/HTML, none mixed), so keeping
  the first copy loses no information. With them removed, the row count
  matches the `ohlc` table exactly (368,251).

In [13]:
#check for repeated ticker/date rows (after cleaning ticker labels)
def detect_duplicate_rows(df):
    """Return every row whose ticker/date pair appears more than once."""
    dup_mask = df.duplicated(subset=["ticker", "ts"], keep=False)
    return df.loc[dup_mask].sort_values(["ticker", "ts"])

### Data Quality Report
Combines every check into one DataFrame with columns
`ticker | issue_type | issue_date | description | severity`.

Duplicates are reported first and then dropped, so they don't create
fake 0% returns or double-count other issues.

In [14]:
def build_data_quality_report(df_hw, df_reference):
    """Diagnose data-quality issues and return an issue-level report."""
    df = prepare_data(df_hw)

    issues = []

    def add(ticker, issue_type, date, description, severity):
        issues.append(make_issue(
            ticker, issue_type, date, description, severity
        ))

    # 0. Duplicate ticker/date rows. Report the extra copies, then drop them
    #    so they don't create fake 0% returns or double-count other issues.
    price_and_volume = ["open", "high", "low", "close", "adj_close", "volume"]
    duplicates = detect_duplicate_rows(df)

    for (ticker, date), group in duplicates.groupby(["ticker", "ts"]):
        identical = len(group[price_and_volume].drop_duplicates()) == 1
        add(
            ticker, "duplicate_row", date,
            f"{len(group)} rows share this ticker/date; values are "
            f"{'identical' if identical else 'DIFFERENT'}. "
            f"{'Extra copy removed.' if identical else 'All copies quarantined.'}",
            "low" if identical else "high",
        )

    df = df.drop_duplicates(subset=["ticker", "ts"], keep="first")

    # A. Missing values in existing rows
    for col in df.columns:
        missing_rows = df.index[df[col].isna()]

        for idx in missing_rows:
            row = df.loc[idx]
            add(
                row.get("ticker", pd.NA),
                "missing_value",
                row.get("ts", pd.NaT),
                f"Column '{col}' has a missing or unparseable value.",
                "medium",
            )

    # B. Missing trading dates relative to ohlc
    gaps = detect_missing_trading_days(df_hw, df_reference)

    for _, gap in gaps.iterrows():
        for date in gap["missing_dates"]:
            add(
                gap["ticker"],
                "missing_trading_day",
                date,
                f"Reference trading date {date.date()} is absent "
                f"from the homework table. "
                f"{gap['summary']}",
                "medium",
            )

    # B2. Unusually long gaps between consecutive dates (no reference needed)
    for _, gap in detect_date_gaps(df).iterrows():
        add(
            gap["ticker"], "date_gap", gap["gap_end"],
            f"{gap['gap_days']} calendar days since previous observation "
            f"on {gap['gap_start'].date()}.",
            "low",
        )

    # Work with valid ticker/date observations for time-series checks
    df = df.dropna(subset=["ticker", "ts"]).copy()
    df = df.sort_values(["ticker", "ts"])

    price_cols = ["open", "high", "low", "close", "adj_close"]

    # C. Nonpositive or non-finite equity prices
    for col in price_cols:
        bad = df[col].notna() & (
            ~np.isfinite(df[col]) | (df[col] <= 0)
        )

        for _, row in df.loc[bad].iterrows():
            add(
                row["ticker"], "invalid_price", row["ts"],
                f"{col} is nonpositive or non-finite: {row[col]!r}.",
                "high",
            )

    # D. Impossible close/high/low relationships
    bad_close = (
        (df["high"] < df["close"])
        | (df["low"] > df["close"])
    ).fillna(False)

    for _, row in df.loc[bad_close].iterrows():
        add(
            row["ticker"], "close_outside_high_low", row["ts"],
            f"Close {row['close']} falls outside the day's "
            f"low/high range [{row['low']}, {row['high']}].",
            "high",
        )

    # E. Full OHLC consistency
    bad_ohlc = (
        (df["low"] > df["high"])
        | (df["open"] < df["low"])
        | (df["open"] > df["high"])
        | (df["close"] < df["low"])
        | (df["close"] > df["high"])
    ).fillna(False)

    for _, row in df.loc[bad_ohlc].iterrows():
        add(
            row["ticker"], "invalid_ohlc_relationship", row["ts"],
            "Expected low <= open <= high and "
            "low <= close <= high.",
            "high",
        )

    # F. Zero-volume days
    zero_volume = df["volume"].eq(0).fillna(False)

    for _, row in df.loc[zero_volume].iterrows():
        add(
            row["ticker"], "zero_volume", row["ts"],
            "Trading volume is zero; check for stale or incomplete data.",
            "medium",
        )

    # G. Daily returns and per-ticker 3-standard-deviation flags
    df["prev_close"] = df.groupby("ticker")["close"].shift(1)

    valid_prev = (
        df["prev_close"].notna()
        & np.isfinite(df["prev_close"])
        & (df["prev_close"] != 0)
        & df["close"].notna()
        & np.isfinite(df["close"])
    )

    df["daily_return"] = np.nan
    df.loc[valid_prev, "daily_return"] = (
        (df.loc[valid_prev, "close"] - df.loc[valid_prev, "prev_close"])
        / df.loc[valid_prev, "prev_close"]
    )

    for ticker, group in df.groupby("ticker"):
        returns = group["daily_return"]
        valid_returns = returns.dropna()

        # ddof=1 is the sample standard deviation
        mean_return = valid_returns.mean()
        std_return = valid_returns.std(ddof=1)

        # Explicitly handle undefined/zero standard deviation.
        if (
            len(valid_returns) < 2
            or not np.isfinite(std_return)
            or std_return == 0
        ):
            continue

        outliers = (
            (returns - mean_return).abs() > 3 * std_return
        ).fillna(False)

        for idx in group.index[outliers]:
            row = df.loc[idx]
            add(
                ticker, "extreme_daily_return", row["ts"],
                f"Daily return {row['daily_return']:.6g}; "
                f"ticker mean={mean_return:.6g}, "
                f"sample std={std_return:.6g}. "
                "Exceeds 3 standard deviations; review, do not "
                "automatically delete.",
                "high",
            )

    report = pd.DataFrame(issues, columns=REPORT_COLUMNS)

    if not report.empty:
        report = report.sort_values(
            ["ticker", "issue_date", "issue_type"],
            na_position="last"
        ).reset_index(drop=True)

    return report

### Run the checks

In [15]:
print("\n" + "=" * 60)
print("MISSING VALUES BY COLUMN")
print("=" * 60)
print(check_missing_values(df))


MISSING VALUES BY COLUMN
           missing_count  missing_percent
adj_close          19855             4.99
ts                     0             0.00
open                   0             0.00
high                   0             0.00
low                    0             0.00
close                  0             0.00
volume                 0             0.00
ticker                 0             0.00


In [16]:
print("\n" + "=" * 60)
print("MISSING TRADING DAYS BY TICKER")
print("=" * 60)

missing_days = detect_missing_trading_days(df, df_reference)

# One row per ticker: how many dates are missing, and over what date range
missing_table = pd.DataFrame({
    "ticker": missing_days["ticker"],
    "missing_dates": missing_days["missing_trading_days"],
    "date_range": (missing_days["start_date"].dt.strftime("%Y-%m-%d")
                   + " to " + missing_days["end_date"].dt.strftime("%Y-%m-%d")),
}).sort_values("ticker").reset_index(drop=True)

display(missing_table)


MISSING TRADING DAYS BY TICKER


,ticker,missing_dates,date_range
0,AAPL,0,1980-12-12 to 2024-10-18
1,AMGN,0,1983-06-17 to 2024-10-18
2,AMZN,0,1997-05-15 to 2024-12-06
3,AXP,0,1972-04-03 to 2024-10-18
4,BA,0,1962-01-02 to 2024-10-18
5,CAT,0,1962-01-02 to 2024-10-18
6,CRM,0,2004-06-23 to 2024-10-18
7,CSCO,0,1990-02-16 to 2024-10-18
8,CVX,0,1962-01-02 to 2024-12-06
9,DIS,0,1962-01-02 to 2024-12-06


In [17]:
print("\n" + "=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)
report = build_data_quality_report(df, df_reference)
print(report["issue_type"].value_counts())
display(report.head(20))


DATA QUALITY REPORT


issue_type
duplicate_row                29460
missing_value                18374
extreme_daily_return          5056
zero_volume                     75
date_gap                        28
invalid_ohlc_relationship        5
invalid_price                    5
Name: count, dtype: int64


,ticker,issue_type,issue_date,description,severity
0,AAPL,extreme_daily_return,1980-12-26,"Daily return 0.0922875; ticker mean=0.0010743,...",high
1,AAPL,duplicate_row,1981-01-02,2 rows share this ticker/date; values are iden...,low
2,AAPL,duplicate_row,1981-01-05,2 rows share this ticker/date; values are iden...,low
3,AAPL,missing_value,1981-01-08,Column 'adj_close' has a missing or unparseabl...,medium
4,AAPL,missing_value,1981-01-13,Column 'adj_close' has a missing or unparseabl...,medium
5,AAPL,duplicate_row,1981-02-10,2 rows share this ticker/date; values are iden...,low
6,AAPL,missing_value,1981-02-12,Column 'adj_close' has a missing or unparseabl...,medium
7,AAPL,duplicate_row,1981-02-27,2 rows share this ticker/date; values are iden...,low
8,AAPL,duplicate_row,1981-03-05,2 rows share this ticker/date; values are iden...,low
9,AAPL,duplicate_row,1981-03-06,2 rows share this ticker/date; values are iden...,low


**How each issue type was detected**

Each row of the report is one issue on one ticker/date. Checks run in this order;
duplicates are reported first and then dropped, so later checks see each day once.

| `issue_type` | Count | How it was found | Severity |
|---|---|---|---|
| `duplicate_row` | 29,460 | After stripping HTML from tickers, `df.duplicated(["ticker", "ts"])` finds ticker/date pairs that appear more than once. One issue per repeated pair. | low if the copies are identical, high if their values differ |
| `missing_value` | 18,374 | `df.isna()` on every column, after duplicates are dropped. All of them are in `adj_close`. | medium |
| `missing_trading_day` | 0 | For each ticker, dates in the `ohlc` reference table (within that ticker's first-to-last date in `ohlc_hw`) that are absent from `ohlc_hw`. | medium |
| `date_gap` | 28 | Gap of more than 5 calendar days between consecutive dates of a ticker (weekends and holidays never exceed 4). Needs no reference table. | low |
| `invalid_price` | 5 | Any of `open`, `high`, `low`, `close`, `adj_close` that is ≤ 0 or not finite. | high |
| `close_outside_high_low` | 0 | `close > high` or `close < low`. | high |
| `invalid_ohlc_relationship` | 5 | Fails `low ≤ open ≤ high` and `low ≤ close ≤ high`. These are the same 5 rows with `open = 0`. | high |
| `zero_volume` | 75 | `volume == 0`, which may mean a stale or incomplete record. | medium |
| `extreme_daily_return` | 5,056 | Daily return `close / previous close − 1`, computed within each ticker. Flagged when it is more than 3 sample standard deviations from that ticker's mean return. Flagged for review, not deleted, since many are real market moves. | high |

Types with a count of 0 were checked but found nothing, so they don't appear in
the `value_counts()` output above.


**Severity levels**

| Severity | Meaning | Issue types |
|---|---|---|
| **High** | The value is wrong or can't be trusted, and using it as-is would distort prices or returns. Needs a repair or a decision before analysis. | `invalid_price`, `close_outside_high_low`, `invalid_ohlc_relationship`, `extreme_daily_return`, `duplicate_row` with *different* values (no way to know which copy is correct) |
| **Medium** | Information is missing or possibly stale, but the surrounding data is sound, so it can be filled or flagged with a standard rule. | `missing_value`, `missing_trading_day`, `zero_volume` |
| **Low** | Harmless or explainable. The fix is mechanical and loses no information. | `duplicate_row` with *identical* values (drop the extra copy), `date_gap` (real market closures, e.g. after 9/11) |

`extreme_daily_return` is high because each one needs review: it could be a data
error, a stock split, or a real market move, and these need different treatments
(see 1.3). It is not deleted automatically.

**Observations from 1.2**
- 29,460 duplicate ticker/date pairs. Every pair is an exact copy, including
  the raw label (26,509 plain/plain, 2,951 HTML/HTML, none mixed), so keeping
  the first copy loses no information. With them removed, the row count
  matches the `ohlc` table exactly (368,251).
- No trading days are missing relative to `ohlc`.
- All 28 date gaps are the 2001-09-10 → 2001-09-17 market closure after 9/11,
  which is a real closure, not missing data.
- 22 tickers end on 2024-10-18, while 8 (AMZN, CVX, DIS, IBM, KO, MRK, NKE, PG)
  run to 2024-12-06.

## 1.3 Clean the Data

What 1.2 found, and how each issue is handled:

| Issue | Handling |
|---|---|
| HTML-wrapped tickers | Stripped with `clean_ticker`; raw → clean mapping kept in `label_audit` |
| Duplicate ticker/date rows | Exact copies: extras removed. Conflicting values: all copies quarantined (none occur) |
| Missing prices / volume | Drop rows with all prices missing *first*; forward-fill prices within ticker; quarantine leading gaps; volume → 0 (flagged) |
| `open = 0` (5 rows) | Previous day's close, clipped into the day's [low, high] |
| Open/close outside [low, high] | Widen high/low to contain them |
| One-day spikes that reverse | Rescale the bar to its neighbors (data error) |
| Missing / corrupt `adj_close` | Factor taken from the matching `ohlc` row; quarantine if none |
| Other >3σ returns | Classify (split vs. market move) and keep, flagged |
| Zero volume | Keep, flagged as possibly stale |

Order matters: raw prices are repaired first, then `adj_close` is recomputed
from the repaired `close`, so the adjustment convention stays consistent.
Every step sets a `flag_*` column so repaired values stay identifiable.


In [18]:
PRICE_COLS = ["open", "high", "low", "close"]

# Every cleaning step records what it changed in a boolean flag column, so
# repaired values stay identifiable in ohlc_hw_cleaned.
FLAG_COLS = [
    "flag_price_filled",      # missing price forward-filled
    "flag_volume_filled",     # missing volume set to 0
    "flag_open_repaired",     # nonpositive / impossible open replaced
    "flag_bounds_repaired",   # high/low widened to contain open and close
    "flag_close_interpolated",  # one-day price spike rescaled to its neighbors
    "flag_adj_repaired",      # adj_close missing or corrupt, rebuilt
    "flag_zero_volume",       # zero volume kept, but flagged as possibly stale
]


def start_cleaning(df):
    """Clean tickers/dates, handle duplicate ticker-date rows, sort, add flags.

    - Exact duplicates (same ticker, date AND values) carry no extra
      information: the extra copies are removed.
    - Conflicting duplicates (same ticker/date, different values) cannot be
      resolved without guessing which copy is right: every copy is quarantined.
    - Rows with no ticker or date cannot be placed in a series: quarantined.

    Returns (clean_df, quarantined_rows).
    """
    out = prepare_data(df)
    key = ["ticker", "ts"]
    value_cols = ["open", "high", "low", "close", "adj_close", "volume"]

    missing_key = out["ticker"].isna() | out["ts"].isna()
    exact_dup = out.duplicated(key + value_cols, keep="first") & ~missing_key
    kept = out.loc[~missing_key & ~exact_dup]
    conflict = kept.duplicated(key, keep=False)

    quarantined = pd.concat([
        out.loc[missing_key].assign(quarantine_reason="ticker or date missing"),
        kept.loc[conflict].assign(quarantine_reason="same ticker/date with conflicting values"),
    ])
    out = kept.loc[~conflict].sort_values(key).reset_index(drop=True)
    for col in FLAG_COLS:
        out[col] = False

    print(f"Exact duplicate rows removed:           {exact_dup.sum():,}")
    print(f"Conflicting duplicate rows quarantined: {conflict.sum():,}")
    print(f"Rows missing ticker/date quarantined:   {missing_key.sum():,}")
    print(f"{len(out):,} rows remain.")
    return out, quarantined


### Handle missing values

In [19]:
def handle_missing_values(df):
    """Handle missing prices and volume, within each ticker.

    - Rows with ALL of open/high/low/close missing are dropped, before filling.
    - Missing prices are forward-filled from the same ticker's last known value.
    - Leading gaps (no earlier value to carry forward) are quarantined with a
      reason, never filled from another ticker.
    - Missing volume is set to 0.

    Returns (repaired_df, quarantined_rows).
    """
    df = df.copy()

    all_missing = df[PRICE_COLS].isna().all(axis=1)
    df = df.loc[~all_missing]

    missing_price = df[PRICE_COLS].isna().any(axis=1)
    df[PRICE_COLS] = df.groupby("ticker")[PRICE_COLS].ffill()
    df["flag_price_filled"] |= missing_price

    leading_gap = df[PRICE_COLS].isna().any(axis=1)
    quarantined = df.loc[leading_gap].assign(
        quarantine_reason="price missing with no earlier value in this ticker")
    df = df.loc[~leading_gap]

    missing_volume = df["volume"].isna()
    df["volume"] = df["volume"].fillna(0)
    df["flag_volume_filled"] |= missing_volume

    print(f"All-prices-missing rows dropped: {all_missing.sum():,}")
    print(f"Rows with a forward-filled price: {missing_price.sum():,}")
    print(f"Leading-gap rows quarantined: {leading_gap.sum():,}")
    print(f"Missing volumes set to 0: {missing_volume.sum():,}")
    return df.reset_index(drop=True), quarantined


### Handle impossible price relationships
**Choice:** a zero open is a recording error, so it gets the previous
day's close (Option 2). For any other open/close outside the day's range,
the day's own prices are trusted and high/low are widened (Option 3).

In [20]:
def repair_price_relationships(df):
    """Fix nonpositive prices and impossible OHLC relationships.

    Choice (documented in the README):
    - A nonpositive OPEN is a recording error. Replace it with the previous
      day's close (Option 2), clipped into today's [low, high]. With no
      previous day, use today's close.
    - Any other nonpositive price is set to NaN and forward-filled.
    - Finally, if open or close still falls outside [low, high], widen the
      range: high = max(open, high, close), low = min(open, low, close)
      (Option 3, using the day's other prices).
    """
    df = df.copy()

    bad_open = df["open"] <= 0
    prev_close = df.groupby("ticker")["close"].shift(1)
    replacement = prev_close.fillna(df["close"]).clip(df["low"], df["high"])
    df.loc[bad_open, "open"] = replacement[bad_open]
    df["flag_open_repaired"] |= bad_open

    for col in ["high", "low", "close"]:
        bad = df[col] <= 0
        df.loc[bad, col] = np.nan
        df[col] = df.groupby("ticker")[col].ffill()
        df["flag_price_filled"] |= bad

    new_high = df[["open", "high", "close"]].max(axis=1)
    new_low = df[["open", "low", "close"]].min(axis=1)
    widened = (new_high != df["high"]) | (new_low != df["low"])
    df["high"], df["low"] = new_high, new_low
    df["flag_bounds_repaired"] |= widened

    print(f"Nonpositive opens replaced: {bad_open.sum():,}")
    print(f"Rows whose high/low were widened: {widened.sum():,}")
    return df

### Investigate extreme returns

**Why there are 5,000+ extreme returns**

For each ticker, a daily return is flagged as extreme when it is more than
3 standard deviations (σ) from that ticker's mean. If returns were normally
distributed, about 0.27% of days (~1,000 of 368,251 rows) would fall beyond 3σ.
The data has 5,056 (~1.4%). This is expected: stock returns are **fat-tailed**,
meaning crashes, earnings surprises and news shocks happen far more often than a
bell curve predicts. A 3σ day is therefore not automatically an error, and
deleting them all would throw away real history. Each one is classified instead.

**1. Data error: a price that jumps and snaps back the next day**

- *Pattern:* one day's bar is far off and the next day returns to the prior
  level (e.g. 4.61 → 9.30 → 4.69). Real stocks don't double for a day and then
  give it all back.
- *Detection* (`repair_one_day_spikes`): the return is beyond 3σ, that day's move
  and the next are both larger than ~35% and in opposite directions, and the next
  close is within 5% of the close before the spike.
- *Treatment:* the whole bar (open/high/low/close) is rescaled so its close equals
  the average of the neighboring closes.
- *Found:* 3 rows, all TRV (1976, 1978, 1981), each recorded at double its true
  value (scale ≈ 0.50).

**2. Split / corporate action: the raw close jumps but `adj_close` doesn't**

- *Pattern:* in a 2-for-1 split each share becomes two, so the price halves
  overnight. The raw return is −50%, but shareholders lost nothing. (AAPL's
  4-for-1 split in August 2020 moved the close from ~$499 to ~$129.)
- *Why `adj_close` is smooth:* adjusted close rescales earlier prices for splits
  and dividends, so the series stays continuous across the split.
- *Detection* (`classify_extreme_returns`): the raw `close` return is beyond 3σ,
  but the `adj_close` return is not.
- *Treatment:* kept and labeled `split_or_corporate_action`. Return calculations
  should use `adj_close`.
- *Found:* 201 rows.

**3. Market move: `adj_close` moves too**

- *Pattern:* a real, large price change, e.g. Black Monday (Oct 19, 1987, when
  the market fell over 20% in one day), the 2008 financial crisis, or the March
  2020 COVID crash and rebound.
- *Detection:* the `adj_close` return is also beyond 3σ, so a split doesn't
  explain it.
- *Treatment:* kept and labeled `market_move`. These days matter most for risk
  models, stress tests and volatility estimates. Removing them would make the
  stocks look less risky than they are.
- *Found:* 4,997 rows.

**Why the order matters**

Data errors are repaired first, then `adj_close` is rebuilt, and only then are
the remaining extreme returns classified. If classification ran first, the TRV
spikes would be mislabeled as market moves. If `adj_close` were rebuilt before
the spikes were fixed, its adjustment factors would come from bad prices.


In [21]:
def repair_one_day_spikes(df, n_std=3.0, min_move=0.30, snap_back=0.05):
    """Repair one-day price spikes that immediately reverse (data errors).

    A day is treated as a DATA ERROR, not a real move, when all of these hold:
    - its close return is more than `n_std` standard deviations from the mean,
    - both its return and the next day's are larger than `min_move` (log ~35%)
      and in opposite directions, and
    - the next close is within `snap_back` (5%) of the close before the spike.
    Real crashes and rebounds (1987, 2008, March 2020) move less than this
    and don't land back exactly where they started, so they are kept.

    The whole bar is rescaled so its close equals the average of the
    neighboring closes, which keeps open/high/low consistent with the close.
    """
    df = df.copy()
    g = df.groupby("ticker")["close"]
    log_ret = np.log(df["close"] / g.shift(1))
    next_ret = log_ret.groupby(df["ticker"]).shift(-1)
    z = (log_ret - log_ret.groupby(df["ticker"]).transform("mean")) / log_ret.groupby(df["ticker"]).transform("std")

    prev_close, next_close = g.shift(1), g.shift(-1)
    spike = (
        (z.abs() > n_std)
        & (log_ret.abs() > min_move)
        & (next_ret.abs() > min_move)
        & (np.sign(log_ret) != np.sign(next_ret))
        & ((next_close / prev_close - 1).abs() < snap_back)
    )

    scale = ((prev_close + next_close) / 2) / df["close"]
    for col in PRICE_COLS:
        df.loc[spike, col] = df.loc[spike, col] * scale[spike]
    df["flag_close_interpolated"] |= spike

    print(f"One-day spikes repaired: {spike.sum():,}")
    if spike.any():
        display(df.loc[spike, ["ticker", "ts", "close"]]
                  .assign(prev_close=prev_close[spike], next_close=next_close[spike],
                          scale_applied=scale[spike].round(3)))
    return df

### Rebuild `adj_close`

In the source data, `adj_close / close` is a factor ≤ 1 that only steps at
dividends and splits. In `ohlc_hw`, about 10,000 values are 4–24× too large
(factor > 1, which is impossible) and about 18,000 are missing.

A bad factor is **not** interpolated or forward-filled: if a dividend or split
fell inside the gap, the guess would be wrong. Instead it is taken from the same
ticker and date in the `ohlc` table, and `adj_close = close × factor` is
recomputed from our repaired `close`. Only the factor comes from `ohlc`; the raw
prices are our own. A row with no valid reference factor would be quarantined
rather than guessed (none occur here; the injected tests show the case).


In [22]:
def load_reference_factors(conn):
    """Date-specific adjustment factors (adj_close / close) from the ohlc table."""
    ref = pd.read_sql_query("SELECT ticker, ts, close, adj_close FROM ohlc", conn,
                            parse_dates=["ts"])
    ref["ticker"] = ref["ticker"].map(clean_ticker)
    ref["ref_factor"] = ref["adj_close"] / ref["close"]
    return ref[["ticker", "ts", "ref_factor"]]


def repair_adj_close(df, ref_factors):
    """Rebuild missing or corrupt adj_close from trusted reference factors.

    adj_close = close * factor, where the factor (adj_close / close) only
    changes at dividends and splits. Backward adjustment can only lower
    historical prices, so a factor above 1 is impossible; a factor more than
    1.5x its 21-day rolling median is also a corrupt value.

    A bad factor is NOT interpolated or forward-filled, because a dividend or
    split inside the gap would make the guess wrong. Instead it is replaced by
    the factor for the same (ticker, ts) in the ohlc table, and adj_close is
    recomputed from the repaired close. Only the factor is taken from ohlc;
    the raw prices stay our own. Rows with no valid reference factor are
    quarantined (removed and returned separately), not guessed.

    Returns (repaired_df, quarantined_rows).
    """
    df = df.copy()
    factor = df["adj_close"] / df["close"]

    above_one = factor > 1.0001
    rolling_median = (factor.where(~above_one)
                      .groupby(df["ticker"])
                      .transform(lambda s: s.rolling(21, center=True, min_periods=3).median()))
    jump = factor / rolling_median > 1.5
    # Days whose close was interpolated also need a fresh factor
    bad = factor.isna() | above_one | jump | df["flag_close_interpolated"]

    ref = df[["ticker", "ts"]].merge(ref_factors, on=["ticker", "ts"], how="left")
    ref_factor = pd.Series(ref["ref_factor"].to_numpy(), index=df.index)
    ref_valid = ref_factor.gt(0) & ref_factor.le(1.0001)

    clean_factor = factor.where(~bad, ref_factor)
    unresolved = bad & ~ref_valid

    df["adj_close"] = df["close"] * clean_factor
    df["flag_adj_repaired"] |= bad & ref_valid

    quarantined = df.loc[unresolved].assign(
        quarantine_reason="adj_close bad and no valid reference factor")
    df = df.loc[~unresolved].reset_index(drop=True)

    print(f"adj_close missing:          {factor.isna().sum():,}")
    print(f"adj_close above close:      {above_one.sum():,}")
    print(f"adj_close jumps vs median:  {(jump & ~above_one).sum():,}")
    print(f"Rebuilt from ohlc factor:   {(bad & ref_valid).sum():,}")
    print(f"Quarantined (no factor):    {unresolved.sum():,}")
    return df, quarantined


In [23]:
def classify_extreme_returns(df, n_std=3.0):
    """Label every remaining >3-standard-deviation close return.

    - split_or_corporate_action: the raw close jumps, but the adjusted price
      doesn't (adj_close absorbs splits), so the raw move isn't economic.
    - market_move: the adjusted price moves too. Real news; kept as-is.
    One-day data errors were already interpolated in the previous step.
    """
    g = df.groupby("ticker")
    raw = np.log(df["close"] / g["close"].shift(1))
    adj = np.log(df["adj_close"] / g["adj_close"].shift(1))

    def zscore(r):
        grp = r.groupby(df["ticker"])
        return (r - grp.transform("mean")) / grp.transform("std")

    raw_extreme = zscore(raw).abs() > n_std
    adj_extreme = zscore(adj).abs() > n_std

    label = pd.Series("", index=df.index)
    label[raw_extreme & ~adj_extreme] = "split_or_corporate_action"
    label[adj_extreme] = "market_move"
    return label

In [24]:
def flag_zero_volume(df):
    """Keep zero-volume days but flag them as possibly stale."""
    df = df.copy()
    df["flag_zero_volume"] = df["volume"].eq(0)
    print(f"Zero-volume days flagged: {df['flag_zero_volume'].sum():,}")
    return df

### Run the cleaning pipeline

In [25]:
ref_factors = load_reference_factors(conn)

clean, key_quarantine = start_cleaning(df)
n_exact_dups = len(df) - len(clean) - len(key_quarantine)
n_after_dedup = len(clean)
clean, gap_quarantine = handle_missing_values(clean)
n_all_missing = n_after_dedup - len(clean) - len(gap_quarantine)
clean = repair_price_relationships(clean)
clean = repair_one_day_spikes(clean)
clean, adj_quarantine = repair_adj_close(clean, ref_factors)
clean = flag_zero_volume(clean)
clean["return_flag"] = classify_extreme_returns(clean)

quarantine = pd.concat([key_quarantine, gap_quarantine, adj_quarantine], ignore_index=True)

print()
print(clean["return_flag"].replace("", "normal").value_counts())




Exact duplicate rows removed:           29,460
Conflicting duplicate rows quarantined: 0
Rows missing ticker/date quarantined:   0
368,251 rows remain.
All-prices-missing rows dropped: 0
Rows with a forward-filled price: 0
Leading-gap rows quarantined: 0
Missing volumes set to 0: 0
Nonpositive opens replaced: 5
Rows whose high/low were widened: 0
One-day spikes repaired: 3


,ticker,ts,close,prev_close,next_close,scale_applied
318332,TRV,1976-10-28,4.650,4.61,4.69,0.504
318873,TRV,1978-12-20,4.055,4.02,4.09,0.505
319538,TRV,1981-08-10,6.030,6.03,6.03,0.500


adj_close missing:          18,374
adj_close above close:      10,348
adj_close jumps vs median:  71
Rebuilt from ohlc factor:   28,793
Quarantined (no factor):    0
Zero-volume days flagged: 75

return_flag
normal                       363053
market_move                    4997
split_or_corporate_action       201
Name: count, dtype: int64


### Row accounting
Every raw row ends up in exactly one place. **Removed** rows carry no
information (exact duplicates, or no prices at all). **Quarantined** rows had a
problem that could not be repaired without guessing; they are kept in
`ohlc_hw_quarantine` with a reason. **Retained** rows are saved to
`ohlc_hw_cleaned`.


In [26]:
# Every raw row ends up in exactly one place: retained, removed, or quarantined
n_conflict = int((key_quarantine["quarantine_reason"] == "same ticker/date with conflicting values").sum())

row_accounting = pd.DataFrame([
    ("removed",     "exact duplicate of another row",                     n_exact_dups),
    ("removed",     "all of open/high/low/close missing",                 n_all_missing),
    ("quarantined", "ticker or date missing",                             len(key_quarantine) - n_conflict),
    ("quarantined", "same ticker/date with conflicting values",           n_conflict),
    ("quarantined", "price missing with no earlier value in this ticker", len(gap_quarantine)),
    ("quarantined", "adj_close bad and no valid reference factor",        len(adj_quarantine)),
    ("retained",    "saved to ohlc_hw_cleaned",                           len(clean)),
], columns=["status", "reason", "rows"])
display(row_accounting)

total = row_accounting["rows"].sum()
print(f"Accounted for {total:,} of {len(df):,} raw rows.")
assert total == len(df), "some rows are unaccounted for"


,status,reason,rows
0,removed,exact duplicate of another row,29460
1,removed,all of open/high/low/close missing,0
2,quarantined,ticker or date missing,0
3,quarantined,same ticker/date with conflicting values,0
4,quarantined,price missing with no earlier value in this ti...,0
5,quarantined,adj_close bad and no valid reference factor,0
6,retained,saved to ohlc_hw_cleaned,368251


Accounted for 397,711 of 397,711 raw rows.


### Validation
The cleaned data must have no missing prices and only valid price relationships.

In [27]:

def validate_clean_data(df):
    """Assert the cleaned data has no missing prices and valid OHLC bounds."""
    price_cols = ["open", "high", "low", "close", "adj_close"]

    missing = df[price_cols].isna().sum()
    nonpositive = (df[price_cols] <= 0).sum()
    invalid_ohlc = (
        (df["low"] > df["high"])
        | (df["open"] < df["low"]) | (df["open"] > df["high"])
        | (df["close"] < df["low"]) | (df["close"] > df["high"])
    )
    duplicates = df.duplicated(["ticker", "ts"])

    print("Missing values per price column:", missing.to_dict())
    print("Nonpositive prices per column:  ", nonpositive.to_dict())
    print("Invalid OHLC relationships:     ", int(invalid_ohlc.sum()))
    print("Duplicate ticker/date rows:     ", int(duplicates.sum()))

    assert missing.sum() == 0, "price columns still have missing values"
    assert nonpositive.sum() == 0, "nonpositive prices remain"
    assert not invalid_ohlc.any(), "invalid OHLC relationships remain"
    assert not duplicates.any(), "duplicate rows remain"
    print("\nAll checks passed.")


validate_clean_data(clean)
    

Missing values per price column: {'open': 0, 'high': 0, 'low': 0, 'close': 0, 'adj_close': 0}
Nonpositive prices per column:   {'open': 0, 'high': 0, 'low': 0, 'close': 0, 'adj_close': 0}
Invalid OHLC relationships:      0
Duplicate ticker/date rows:      0

All checks passed.


### Before / after statistics

In [28]:

def daily_abs_return_max(frame, col):
    """Largest absolute one-day % change of `col`, within each ticker."""
    frame = frame.sort_values(["ticker", "ts"])
    return frame.groupby("ticker")[col].pct_change(fill_method=None).abs().max()


def quality_stats(frame):
    """Data-quality summary statistics for one DataFrame (any set of tickers)."""
    return {
        "rows": len(frame),
        "duplicate ticker/date rows": int(frame.duplicated(["ticker", "ts"]).sum()),
        "missing adj_close": int(frame["adj_close"].isna().sum()),
        "adj_close > close": int((frame["adj_close"] > frame["close"] * 1.0001).sum()),
        "nonpositive open": int((frame["open"] <= 0).sum()),
        "max adj_close": frame["adj_close"].max(),
        "max 1-day adj_close change": daily_abs_return_max(
            frame.drop_duplicates(["ticker", "ts"]), "adj_close"),
    }


def before_after_table(before, after):
    """Side-by-side before/after statistics, formatted for display."""
    table = pd.DataFrame({"before": quality_stats(before), "after": quality_stats(after)})
    return table.apply(lambda col: col.map("{:,.2f}".format))


raw = prepare_data(df)  # cleaned tickers only; still has every other issue

print("ALL STOCKS")
display(before_after_table(raw, clean))

# One before/after table per stock
for ticker in sorted(clean["ticker"].unique()):
    print(f"\n{ticker}")
    display(before_after_table(raw[raw["ticker"] == ticker],
                               clean[clean["ticker"] == ticker]))

price_cols = ["open", "high", "low", "close", "adj_close", "volume"]
print("Before:")
display(raw[price_cols].describe())
print("After:")
display(clean[price_cols].describe())


ALL STOCKS


,before,after
rows,"397,711.00","368,251.00"
duplicate ticker/date rows,"29,460.00",0.00
missing adj_close,"19,855.00",0.00
adj_close > close,"11,189.00",0.00
nonpositive open,5.00,0.00
max adj_close,"10,442.51",605.40
max 1-day adj_close change,24.93,0.76



AAPL


,before,after
rows,"11,942.00","11,050.00"
duplicate ticker/date rows,892.00,0.00
missing adj_close,617.00,0.00
adj_close > close,363.00,0.00
nonpositive open,0.00,0.00
max adj_close,"5,109.15",235.00
max 1-day adj_close change,23.79,0.52



AMGN


...

,before,after
rows,"14,295.00","13,252.00"
duplicate ticker/date rows,"1,043.00",0.00
missing adj_close,661.00,0.00
adj_close > close,420.00,0.00
nonpositive open,0.00,0.00
max adj_close,"1,577.08",81.65
max 1-day adj_close change,23.84,0.17


Before:


,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


After:


,open,high,low,close,adj_close,volume
count,368251.000000,368251.000000,368251.000000,368251.000000,368251.000000,3.682510e+05
mean,46.308741,46.771165,45.839728,46.316672,37.274152,3.095142e+07
std,67.143027,67.789091,66.489226,67.154713,63.587595,1.266145e+08
min,0.005208,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.210000,4.250000,4.170000,4.210000,1.310000,1.731400e+06
50%,19.890000,20.090000,19.670000,19.900000,11.460000,4.649600e+06
75%,57.560000,58.160000,56.990000,57.570000,41.570000,1.102151e+07
max,604.260000,608.630000,597.880000,605.400000,605.400000,9.276606e+09


### Sanity check against the `ohlc` table
The `ohlc` table holds the same prices without the injected problems.
Apart from supplying the `adj_close` adjustment factors, it was not used to
clean anything; comparing against it shows how well the other rules worked.

In [29]:

ref_full = pd.read_sql_query("SELECT * FROM ohlc", conn, parse_dates=["ts"])
compare = clean.merge(ref_full, on=["ticker", "ts"], suffixes=("", "_ref"))

for col in ["open", "close", "adj_close"]:
    rel_err = (compare[col] / compare[f"{col}_ref"] - 1).abs()
    print(f"{col:9s}: {(rel_err > 0.01).sum():4d} rows differ by more than 1% "
            f"(of {len(compare):,})")


open     :    8 rows differ by more than 1% (of 368,251)
close    :    3 rows differ by more than 1% (of 368,251)
adj_close:    3 rows differ by more than 1% (of 368,251)


`open` differs only on the 5 repaired zero opens and the 3 repaired
one-day spikes, and `close` only on those 3 spikes (TRV, where the
reference table still has the doubled bar). `adj_close` differs only on the
same 3 TRV days.

### Save `ohlc_hw_cleaned`

In [30]:
to_save = clean.copy()
to_save["ts"] = to_save["ts"].dt.strftime("%Y-%m-%d")  # same format as ohlc_hw
to_save.to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)
conn.execute("CREATE INDEX IF NOT EXISTS idx_ohlc_hw_cleaned_ticker_ts "
             "ON ohlc_hw_cleaned (ticker, ts)")

# Quarantined rows are kept in their own table, with a reason for each
quarantine_to_save = quarantine.copy()
quarantine_to_save["ts"] = quarantine_to_save["ts"].dt.strftime("%Y-%m-%d")
quarantine_to_save.to_sql("ohlc_hw_quarantine", conn, if_exists="replace", index=False)

conn.commit()

saved = pd.read_sql_query("SELECT COUNT(*) AS n, COUNT(DISTINCT ticker) AS tickers "
                          "FROM ohlc_hw_cleaned", conn)
print(f"Saved ohlc_hw_cleaned: {saved.loc[0, 'n']:,} rows, "
      f"{saved.loc[0, 'tickers']} tickers")
print(f"Saved ohlc_hw_quarantine: {len(quarantine_to_save):,} rows")

    

Saved ohlc_hw_cleaned: 368,251 rows, 30 tickers
Saved ohlc_hw_quarantine: 0 rows


### Tests on injected examples

Several required checks found **zero** issues in `ohlc_hw`: no missing trading
days, no close outside the high/low range, no missing volume, no rows with
every price missing, no leading gaps, no unresolvable `adj_close` and no
conflicting duplicates. A detector that finds nothing could be broken, so here
each one runs on a tiny made-up ticker with those problems injected on purpose.

| Date | Injected problem | Expected result |
|---|---|---|
| 2024-01-02 | first `close` missing (nothing earlier to fill from) | quarantined as a leading gap |
| 2024-01-03 | `close` 11.0 above `high` 10.6 | flagged; high widened to 11.0 |
| 2024-01-04 | `volume` missing | flagged; set to 0 with `flag_volume_filled` |
| 2024-01-05 | row deleted | flagged as `missing_trading_day` |
| 2024-01-08 | all of open/high/low/close missing | row dropped *before* filling |
| 2024-01-09 | `close` missing, and no reference `adj_close` factor | close forward-filled, bounds rechecked; then quarantined (no factor) |

The 2024-01-08 row is dropped rather than filled: with all four raw prices
missing, forward-filling would invent a complete OHLC bar (and a fake 0% return)
from the previous day. Filling is only used when a row has some real prices.
The drop happens *before* filling; otherwise the row would no longer look
empty and would be kept.

A separate test below covers duplicates: an exact copy is removed, and a
conflicting pair (same date, different close) is quarantined.

In [31]:
# Six consecutive trading days for a made-up ticker. The reference calendar
# has all six; the homework-style table has problems injected on purpose.
dates = pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04",
                        "2024-01-05", "2024-01-08", "2024-01-09"])
toy_reference = pd.DataFrame({"ts": dates, "ticker": "TEST"})

toy = pd.DataFrame({
    "ts":        dates,
    "open":      [10.0, 10.2, 10.1, 10.3, np.nan, 10.6],
    "high":      [10.5, 10.6, 10.5, 10.7, np.nan, 10.9],
    "low":       [ 9.8, 10.0,  9.9, 10.1, np.nan, 10.5],
    "close":     [np.nan, 11.0, 10.3, 10.4, np.nan, np.nan],
    "adj_close": [ 9.18, 9.90, 9.27, 9.36, np.nan, np.nan],
    "volume":    [1000, 1200, np.nan, 1100, 900, 1300],
    "ticker":    "TEST",
})
toy = toy[toy["ts"] != "2024-01-05"].reset_index(drop=True)  # delete one trading day
display(toy)


,ts,open,high,low,close,adj_close,volume,ticker
0,2024-01-02,10.0,10.5,9.8,NaN,9.18,1000.0,TEST
1,2024-01-03,10.2,10.6,10.0,11.0,9.90,1200.0,TEST
2,2024-01-04,10.1,10.5,9.9,10.3,9.27,NaN,TEST
3,2024-01-08,NaN,NaN,NaN,NaN,NaN,900.0,TEST
4,2024-01-09,10.6,10.9,10.5,NaN,NaN,1300.0,TEST


In [32]:
# Reference factors for the toy ticker. 2024-01-09 has none, so its missing
# adj_close cannot be rebuilt and the row must be quarantined, not guessed.
toy_ref_factors = pd.DataFrame({
    "ticker": "TEST",
    "ts": dates[dates != "2024-01-09"],
    "ref_factor": 0.9,
})

toy_clean, toy_key_quarantine = start_cleaning(toy)
toy_clean, toy_gap_quarantine = handle_missing_values(toy_clean)
toy_clean = repair_price_relationships(toy_clean)
toy_clean, toy_adj_quarantine = repair_adj_close(toy_clean, toy_ref_factors)
toy_quarantine = pd.concat([toy_key_quarantine, toy_gap_quarantine, toy_adj_quarantine], ignore_index=True)


display(toy_clean[["ts", "open", "high", "low", "close", "adj_close", "volume",
                   "flag_price_filled", "flag_volume_filled",
                   "flag_bounds_repaired", "flag_adj_repaired"]])
print("Quarantined:")
display(toy_quarantine[["ts", "close", "adj_close", "quarantine_reason"]])

assert pd.Timestamp("2024-01-08") not in set(toy_clean["ts"]), "all-missing row was kept"
assert toy_clean.loc[toy_clean["ts"] == "2024-01-04", "volume"].item() == 0
assert list(toy_quarantine["ts"]) == [pd.Timestamp("2024-01-02"), pd.Timestamp("2024-01-09")], \
    "leading gap or unresolved adj_close was not quarantined"
validate_clean_data(toy_clean)





Exact duplicate rows removed:           0
Conflicting duplicate rows quarantined: 0
Rows missing ticker/date quarantined:   0
5 rows remain.
All-prices-missing rows dropped: 1
Rows with a forward-filled price: 2
Leading-gap rows quarantined: 1
Missing volumes set to 0: 1
Nonpositive opens replaced: 0
Rows whose high/low were widened: 2
adj_close missing:          1
adj_close above close:      0
adj_close jumps vs median:  0
Rebuilt from ohlc factor:   0
Quarantined (no factor):    1


,ts,open,high,low,close,adj_close,volume,flag_price_filled,flag_volume_filled,flag_bounds_repaired,flag_adj_repaired
0,2024-01-03,10.2,11.0,10.0,11.0,9.90,1200.0,False,False,True,False
1,2024-01-04,10.1,10.5,9.9,10.3,9.27,0.0,False,True,False,False


Quarantined:


,ts,close,adj_close,quarantine_reason
0,2024-01-02,NaN,9.18,price missing with no earlier value in this ti...
1,2024-01-09,10.3,NaN,adj_close bad and no valid reference factor


Missing values per price column: {'open': 0, 'high': 0, 'low': 0, 'close': 0, 'adj_close': 0}
Nonpositive prices per column:   {'open': 0, 'high': 0, 'low': 0, 'close': 0, 'adj_close': 0}
Invalid OHLC relationships:      0
Duplicate ticker/date rows:      0

All checks passed.


In [33]:
# Duplicate handling: 01-02 appears twice with identical values (one copy is
# removed); 01-03 appears twice with different closes (both are quarantined).
dup_test = pd.DataFrame({
    "ts":        pd.to_datetime(["2024-01-02", "2024-01-02", "2024-01-03", "2024-01-03"]),
    "open":      [10.0, 10.0, 10.0, 10.0],
    "high":      [11.0, 11.0, 11.0, 11.0],
    "low":       [ 9.0,  9.0,  9.0,  9.0],
    "close":     [10.5, 10.5, 10.5, 10.8],
    "adj_close": [9.45, 9.45, 9.45, 9.72],
    "volume":    [100, 100, 100, 100],
    "ticker":    "TEST",
})
dup_clean, dup_quarantine = start_cleaning(dup_test)
display(dup_quarantine[["ts", "close", "quarantine_reason"]])

assert list(dup_clean["ts"]) == [pd.Timestamp("2024-01-02")], "exact duplicate not collapsed to one row"
assert len(dup_quarantine) == 2, "conflicting duplicates were not quarantined"


Exact duplicate rows removed:           1
Conflicting duplicate rows quarantined: 2
Rows missing ticker/date quarantined:   0
1 rows remain.


,ts,close,quarantine_reason
2,2024-01-03,10.5,same ticker/date with conflicting values
3,2024-01-03,10.8,same ticker/date with conflicting values


In [34]:
conn.close()